In [ ]:
# Load the data from kaggle and upload here.
!unzip /content/super-ai-engineer-s-6-fah-mai-rag-challenge-level-1.zip

In [ ]:
!pip install -q sentence-transformers pythainlp rank-bm25 requests python-dotenv

In [ ]:
# === CONFIGURATION ===
# Change N_QUESTIONS to 100 for a full competition run.

N_QUESTIONS = 100  # 10 for demo, 100 for real submission
DATA_DIR = "/content/data"
KB_DIR = f"{DATA_DIR}/knowledge_base"

In [ ]:
import os, csv, re, time, requests
import numpy as np
from pathlib import Path
from google.colab import userdata

THAILLM_API_KEY = userdata.get('ThaiLLM')

In [ ]:
def ask_llm(messages, model="kbtg", max_retries=5):
    """Call ThaiLLM API with retry and rate-limit handling.

    Available models: typhoon, openthaigpt, pathumma, kbtg
    """
    url = f"http://thaillm.or.th/api/{model}/v1/chat/completions"
    headers = {"Content-Type": "application/json", "apikey": THAILLM_API_KEY}
    payload = {
        "model": "/model",
        "messages": messages,
        "max_tokens": 2024,
        "temperature": 0,
    }

    for attempt in range(max_retries):
        try:
            resp = requests.post(url, headers=headers, json=payload, timeout=120)

            # Rate limit — wait and retry
            if resp.status_code == 429:
                wait = min(2 ** attempt, 30)
                print(f"  Rate limited, waiting {wait}s...")
                time.sleep(wait)
                continue

            resp.raise_for_status()
            return resp.json()["choices"][0]["message"]["content"].strip()

        except requests.exceptions.RequestException as e:
            wait = 2 ** attempt
            print(f"  Error: {e}, retrying in {wait}s...")
            time.sleep(wait)

    return None


def parse_answer(text):
    """Extract answer number from LLM response."""
    if text is None:
        return None
    # Remove any <think>...</think> blocks (some models do chain-of-thought)
    clean = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    # Look for ANSWER: X pattern
    m = re.search(r"ANSWER:\s*(\d+)", clean)
    if m:
        return int(m.group(1))
    # Fallback: first standalone number 1-10
    for d in re.findall(r"\b(\d{1,2})\b", clean):
        if 1 <= int(d) <= 10:
            return int(d)
    return None

In [ ]:
# Ask without context — LLM has no idea about FahMai's products
response = ask_llm([
    {"role": "user", "content": "Watch S3 Ultra กันน้ำได้กี่ ATM ครับ?"}
])
print("LLM response (no context):", response)
print("\n→ The LLM doesn't know FahMai-specific facts. We need RAG!")

In [ ]:
questions = []
with open(f"{DATA_DIR}/questions.csv", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        choices = {str(i): row[f"choice_{i}"] for i in range(1, 11)}
        questions.append({"id": int(row["id"]), "question": row["question"], "choices": choices})

print(f"Loaded {len(questions)} questions, using first {N_QUESTIONS} for this run")
print(f"\nExample — Q1: {questions[0]['question']}")
for k, v in questions[0]["choices"].items():
    print(f"  {k}. {v}")

In [ ]:
kb_dir = Path(KB_DIR)
documents = []
for fp in sorted(kb_dir.rglob("*.md")):
    text = fp.read_text(encoding="utf-8")
    documents.append({"path": str(fp.relative_to(kb_dir)), "text": text})

print(f"Loaded {len(documents)} documents")
print(f"  products/: {sum(1 for d in documents if 'products/' in d['path'])}")
print(f"  policies/: {sum(1 for d in documents if 'policies/' in d['path'])}")
print(f"  store_info/: {sum(1 for d in documents if 'store_info/' in d['path'])}")

# Preview one document
print(f"\n--- Sample: {documents[0]['path']} ---")
print(documents[0]["text"][:500])

In [ ]:
CHUNK_SIZE = 900
CHUNK_OVERLAP = 100

def make_chunks(text, size, overlap):
    """Split text into overlapping fixed-size windows."""
    if len(text) <= size:
        return [text]
    chunks = []
    start = 0
    while start < len(text):
        chunks.append(text[start : start + size])
        start += size - overlap
    return chunks

# Build all chunks
chunks = []
for doc in documents:
    for window in make_chunks(doc["text"], CHUNK_SIZE, CHUNK_OVERLAP):
        chunks.append({"text": window, "source": doc["path"]})

print(f"Created {len(chunks)} chunks from {len(documents)} documents")
print(f"\n--- Sample chunk ---")
print(f"Source: {chunks[0]['source']}")
print(chunks[1]["text"])

In [ ]:
from google.colab import userdata
from sentence_transformers import SentenceTransformer
import torch

# HF_TOKEN is not strictly needed for public models
# HF_TOKEN = userdata.get('HF_TOKEN') # Keeping this line commented for now

# Using 'distiluse-base-multilingual-cased-v2' as a strong multilingual alternative
device = 'cuda' if torch.cuda.is_available() else 'cpu'
embed_model = SentenceTransformer("intfloat/multilingual-e5-large", device = device)

# Embed all chunks
chunk_texts = [c["text"] for c in chunks]
chunk_embeddings = embed_model.encode(chunk_texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)

print(f"Chunk embeddings shape: {chunk_embeddings.shape}")  # (n_chunks, 1024) or other dimension depending on model

In [ ]:
import numpy as np
from sentence_transformers import CrossEncoder

RETRIEVE_K = 10 # Number of chunks to retrieve initially with dense retrieval
FINAL_K = 5   # Number of chunks to keep after reranking
RERANK_MODEL_NAME = "BAAI/bge-reranker-v2-m3"

# Load the reranker model
reranker = CrossEncoder(RERANK_MODEL_NAME)

def dense_retrieve(query, chunk_embs, k=RETRIEVE_K):
    """Return indices of top-k most similar chunks to the query using dense retrieval."""
    q_emb = embed_model.encode([query], normalize_embeddings=True)
    scores = np.dot(chunk_embs, q_emb.T).flatten()  # cosine similarity (vectors are normalized)
    top_idx = np.argsort(scores)[::-1][:k]
    return top_idx, scores[top_idx]

def rerank_and_select_chunks(query, candidate_chunk_indices, k=FINAL_K):
    """Rerank candidate chunks using a CrossEncoder and return the top-k chunk objects."""
    if candidate_chunk_indices.size == 0: # Fixed: Check if numpy array is empty
        return []

    # Prepare sentence pairs for the reranker
    sentence_pairs = [[query, chunks[idx]['text']] for idx in candidate_chunk_indices]

    # Get reranker scores
    rerank_scores = reranker.predict(sentence_pairs)

    # Sort chunks by reranker scores
    reranked_indices_with_scores = sorted(
        zip(candidate_chunk_indices, rerank_scores),
        key=lambda x: x[1],
        reverse=True
    )

    # Select top-k chunk objects
    top_k_chunk_indices = [idx for idx, _ in reranked_indices_with_scores[:k]]
    return [chunks[idx] for idx in top_k_chunk_indices]


# Demo: retrieve for Q1 (using initial dense retrieval for demonstration)
q = questions[0]
idx, scores = dense_retrieve(q["question"], chunk_embeddings, k=RETRIEVE_K)

print(f"Question: {q['question']}\n")
print(f"Initial Dense Retrieval (top {RETRIEVE_K} candidates):\n")
for rank, (i, s) in enumerate(zip(idx, scores), 1):
    print(f"  Rank {rank} (score={s:.3f}) [{chunks[i]['source']}]")
    print(f"  {chunks[i]['text'][:150]}...")
    print()

# Demo: Reranked Retrieval for Q1
retrieved_reranked = rerank_and_select_chunks(q["question"], idx, k=FINAL_K)
print(f"Reranked Retrieval (top {FINAL_K} after reranking):\n")
for rank, chunk_obj in enumerate(retrieved_reranked, 1):
    print(f"  Rank {rank} [{chunk_obj['source']}]")
    print(f"  {chunk_obj['text'][:150]}...")
    print()


In [ ]:
# An improved system prompt based on RAG best practices for multiple-choice questions
SYSTEM_PROMPT = """
คุณคือ AI Customer Service ของร้าน FahMai

หน้าที่:
- ตอบคำถามโดยใช้เฉพาะข้อมูลจาก context ที่ให้มาเท่านั้น
- ห้ามใช้ความรู้ภายนอกหรือเดาเด็ดขาด

รูปแบบคำตอบ:
- เลือกคำตอบที่ถูกต้องที่สุดเพียง 1 ข้อ จากตัวเลือก 1–10
- ตอบในรูปแบบ: ANSWER: X
- ห้ามอธิบาย ห้ามมีข้อความอื่น

กฎการตัดสินใจ:
1. ถ้ามีข้อมูลใน context ที่ตรงกับคำถาม → เลือกคำตอบที่ตรงที่สุด
2. ถ้าไม่มีข้อมูลใน context → ตอบ 9
3. ถ้าคำถามไม่เกี่ยวข้องกับสินค้า/บริการ → ตอบ 10

วิธีคิด:
- เปรียบเทียบคำถามกับตัวเลือกทีละข้อ
- ใช้ข้อมูลใน context เพื่อหา "ตัวเลือกที่ตรงที่สุด"
- ห้ามเลือกคำตอบถ้าไม่มีหลักฐานใน context

จำไว้: ต้องมีหลักฐานใน context เท่านั้น
"""

def build_rag_prompt(question, choices, retrieved_chunks):
    # จำกัด chunk (สำคัญมาก)
    retrieved_chunks = retrieved_chunks[:5]

    # จัด context ให้อ่านง่าย + มี label
    context_parts = []
    for i, c in enumerate(retrieved_chunks):
        context_parts.append(f"[เอกสารที่ {i+1}]\n{c['text']}")
    context = "\n\n".join(context_parts)

    # จัด choices
    choices_text = "\n".join(f"{k}) {v}" for k, v in choices.items())

    return f"""
[ข้อมูลจากฐานความรู้]
{context}

====================

[คำถาม]
{question}

====================

[ตัวเลือก]
{choices_text}
9) ไม่มีข้อมูลนี้ในฐานข้อมูล
10) คำถามนี้ไม่เกี่ยวข้อง

====================

คำสั่ง:
- พิจารณาข้อมูลจาก context เท่านั้น
- เปรียบเทียบคำถามกับตัวเลือกทีละข้อ
- เลือกคำตอบที่ "ตรงที่สุด" และมีหลักฐานใน context

ตอบในรูปแบบ:
ANSWER: X
"""

# Demo: answer Q1
q = questions[0]
idx, _ = dense_retrieve(q["question"], chunk_embeddings)
retrieved = [chunks[i] for i in idx]

prompt = build_rag_prompt(q["question"], q["choices"], retrieved)
raw = ask_llm([
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": prompt},
])
answer = parse_answer(raw)
print(f"Q{q['id']}: {q['question']}")
print(f"LLM raw: {raw}")
print(f"Parsed answer: {answer}")

In [ ]:
def run_pipeline(questions, retrieve_fn, label="dense", n=N_QUESTIONS):
    """Run retrieval + LLM on first n questions. Returns predictions dict."""
    predictions = {}
    for i, q in enumerate(questions[:n]):
        retrieved_chunks = retrieve_fn(q["question"])
        prompt = build_rag_prompt(q["question"], q["choices"], retrieved_chunks)
        raw = ask_llm([
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ])
        pred = parse_answer(raw)
        predictions[q["id"]] = pred if pred else 1  # default to 1 if parse fails
        print(f"  Q{q['id']:>3}: pred={predictions[q['id']]}")
        time.sleep(0.3)  # be nice to the API
    print(f"\n{label}: answered {len(predictions)} questions")
    return predictions

# Dense retrieval function
def dense_retrieve_chunks(query):
    idx, _ = dense_retrieve(query, chunk_embeddings)
    return [chunks[i] for i in idx]

dense_preds = run_pipeline(questions, dense_retrieve_chunks, label="dense")

In [ ]:
# Use dense predictions as the submission (change to hybrid_preds or bm25_preds to try others)
best_preds = dense_preds

with open("submission.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "answer"])
    for q in questions:
        qid = q["id"]
        writer.writerow([qid, best_preds.get(qid, 1)])  # default=1 for unanswered

print(f"submission.csv written ({len(questions)} rows)")